## Feature generation

In [1]:
import pandas as pd
import numpy as np
import re
from rdkit.Chem.Descriptors import ExactMolWt
from rdkit import Chem
from rdkit.Chem import QED, Descriptors, EState, GraphDescriptors, MolSurf
import skfp.fingerprints as sfp

In [2]:
visco = pd.read_csv('data/viscoelastic_cleared.csv', keep_default_na=False, na_values='')

In [3]:
visco.info()

<class 'pandas.DataFrame'>
RangeIndex: 8152 entries, 0 to 8151
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_surfactant1      8152 non-null   str    
 1   type_surfactant1    8152 non-null   str    
 2   smiles_surfactant1  8152 non-null   str    
 3   c_surfactant1       8152 non-null   float64
 4   id_surfactant2      8152 non-null   str    
 5   type_surfactant2    8152 non-null   str    
 6   smiles_surfactant2  8152 non-null   str    
 7   c_surfactant2       8152 non-null   float64
 8   id_additive1        8152 non-null   str    
 9   smiles_additive1    8152 non-null   str    
 10  c_additive1         8152 non-null   float64
 11  id_additive2        8152 non-null   str    
 12  smiles_additive2    8152 non-null   str    
 13  c_additive2         8152 non-null   float64
 14  temperature         8152 non-null   int64  
 15  viscoelasticity     8152 non-null   float64
 16  g_0              

In [4]:
visco.head()

,id_surfactant1,type_surfactant1,smiles_surfactant1,c_surfactant1,id_surfactant2,type_surfactant2,smiles_surfactant2,c_surfactant2,id_additive1,smiles_additive1,...,id_additive2,smiles_additive2,c_additive2,temperature,viscoelasticity,g_0,g_min,omega_cr,tau_r,eta_0
0,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,0.0,25,0.0,NaN,NaN,NaN,NaN,0.02
1,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,0.0,25,1.0,4.0,2.70,150.0,0.01,0.03
2,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,0.0,25,1.0,4.5,1.20,10.0,0.10,0.45
3,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,0.0,25,1.0,5.0,0.50,0.8,1.25,6.25
4,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,0.0,25,1.0,4.2,0.35,1.5,0.67,2.80


In [5]:
smiles = visco['smiles_surfactant1']
smiles = pd.concat([smiles, visco['smiles_surfactant2']])
smiles = pd.concat([smiles, visco['smiles_additive1']])
smiles = pd.concat([smiles, visco['smiles_additive2']])
smiles = smiles.drop_duplicates()
smiles = smiles.replace('None', np.nan).dropna()
len(smiles)

137

In [6]:
# Tail length
def extract_tail_c_number(smiles: str):
    # For now just count n of leading C's
    if smiles == 'None':
        return np.nan
    n_c = re.match(r"^(C+)", smiles)
    return len(n_c.group(1)) if n_c else 0


# Surfactant type is already in the data but further more advance type can be added later(look in README)

def extract_molecular_weight(smiles: str):
    if smiles == 'None':
        return np.nan
    mol = Chem.MolFromSmiles(smiles)
    return ExactMolWt(mol)


def generate_chen(smiles: str):
    if smiles == 'None':
        return pd.Series({
            "chen_MolLogP": np.nan,
            "chen_Chi1v": np.nan,
            "chen_VSA_EState7": np.nan,
            "chen_VSA_EState8": np.nan,
            "chen_qed": np.nan,
            "chen_FpDensityMorgan1": np.nan,
            "chen_SMR_VSA5": np.nan,
            "chen_PEOE_VSA6": np.nan,
            "chen_MinPartialCharge": np.nan,
            "chen_BertzCT": np.nan,
            "chen_VSA_EState1": np.nan,
            "chen_MaxAbsEStateIndex": np.nan,
            "chen_Balaban": np.nan,
            "chen_SlogP_VSA2": np.nan,
            "chen_HallKierAlpha": np.nan,
            "chen_EState_VSA5": np.nan,
            "chen_MaxPartialCharge": np.nan,
            "chen_PEOE_VSA7": np.nan,
            "chen_PEOE_VSA8": np.nan
        })

    mol = Chem.MolFromSmiles(smiles)

    return pd.Series({
        "chen_MolLogP": Descriptors.MolLogP(mol),
        "chen_Chi1v": GraphDescriptors.Chi1v(mol),
        "chen_VSA_EState7": EState.EState_VSA.EState_VSA7(mol),
        "chen_VSA_EState8": EState.EState_VSA.EState_VSA8(mol),
        "chen_qed": QED.qed(mol),
        "chen_FpDensityMorgan1": Descriptors.FpDensityMorgan1(mol),
        "chen_SMR_VSA5": MolSurf.SMR_VSA5(mol),
        "chen_PEOE_VSA6": MolSurf.PEOE_VSA6(mol),
        "chen_MinPartialCharge": Descriptors.MinPartialCharge(mol),
        "chen_BertzCT": GraphDescriptors.BertzCT(mol),
        "chen_VSA_EState1": EState.EState_VSA.EState_VSA1(mol),
        "chen_MaxAbsEStateIndex": EState.MaxAbsEStateIndex(mol),
        "chen_Balaban": GraphDescriptors.BalabanJ(mol),
        "chen_SlogP_VSA2": MolSurf.SlogP_VSA2(mol),
        "chen_HallKierAlpha": Descriptors.HallKierAlpha(mol),
        "chen_EState_VSA5": EState.EState_VSA.EState_VSA5(mol),
        "chen_MaxPartialCharge": Descriptors.MaxPartialCharge(mol),
        "chen_PEOE_VSA7": MolSurf.PEOE_VSA7(mol),
        "chen_PEOE_VSA8": MolSurf.PEOE_VSA8(mol)
    })

# Requires .transform() - due  to representation it's not really suitable to put this into pandas(numpy better?)
# features = sfp.PhysiochemicalPropertiesFingerprint(fp_size=256)

In [7]:
expert_chen = pd.DataFrame(smiles, columns=['smiles'])

In [8]:
expert_chen.head()

,smiles
0,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+]
8,CCCCCCCCCCCCCCCC[N+](C)(C)CCCS(=O)(=O)[O-]
11,CCCCCCCCCCCCCC[N+](C)(C)CCCS(=O)(=O)[O-]
13,CCCCCCCCCCCCCCCC[N+](C)(C)C.Cc1ccc(S(=O)(=O)[O...
87,CCCCCCCCCCCCOCCOCCOCCOS(=O)(=O)[O-].[Na+]


In [9]:
chen_columns = [
    "chen_MolLogP",
    "chen_Chi1v",
    "chen_VSA_EState7",
    "chen_VSA_EState8",
    "chen_qed",
    "chen_FpDensityMorgan1",
    "chen_SMR_VSA5",
    "chen_PEOE_VSA6",
    "chen_MinPartialCharge",
    "chen_BertzCT",
    "chen_VSA_EState1",
    "chen_MaxAbsEStateIndex",
    "chen_Balaban",
    "chen_SlogP_VSA2",
    "chen_HallKierAlpha",
    "chen_EState_VSA5",
    "chen_MaxPartialCharge",
    "chen_PEOE_VSA7",
    "chen_PEOE_VSA8"
]

In [10]:
expert_chen[chen_columns] = expert_chen.apply(
    lambda row: generate_chen(row['smiles']),
    axis=1,
    result_type='expand'
)

expert_chen['molecular_weight'] = expert_chen['smiles'].apply(extract_molecular_weight)
expert_chen['tail_length'] = expert_chen['smiles'].apply(extract_tail_c_number)

In [12]:
expert_chen.to_csv('data/expert_chen.csv', index=False)